<a href="https://colab.research.google.com/github/sangalo20/kubectl-ai-kubernetes/blob/main/Kubectl_ai_on_Kubernetes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Introduction

[**kubectl-ai**](https://github.com/GoogleCloudPlatform/kubectl-ai) is an AI-powered Kubernetes assistant that translates natural language into precise Kubernetes operations — making cluster management more accessible and efficient.

This is a **hands-on workshop notebook**: every step is a runnable cell, so you can follow along live and run everything from this Colab.

**What you'll do**

- Connect this Colab to your own Google Cloud project
- Create a GKE cluster and set up kubectl-ai
- Manage, create, patch, and troubleshoot Kubernetes resources using plain English

**What you'll learn**

- How to drive a real GKE cluster with natural language queries
- How kubectl-ai translates your intent into `kubectl` commands
- Single-shot vs. interactive modes of kubectl-ai

**What you'll need**

- ✅ A Google account
- ✅ A Google Cloud project **with billing enabled**
- ✅ A Gemini API key (free from [Google AI Studio](https://aistudio.google.com/apikey) — we'll set it up in Section 4)

> Basic familiarity with Kubernetes concepts (pods, deployments, namespaces) is assumed.

**How to follow along:** run each code cell in order with `Shift+Enter`. Cells marked 🏋️ are exercises for you to try yourself.

# 2. Connect Colab to your Google Cloud project

First, authenticate this Colab session with your Google account, then point it at your Cloud project.

> 💡 If you don't have a project yet, create one in the [Cloud Console](https://console.cloud.google.com/projectcreate) and make sure **billing is enabled**.

Run the next cell and complete the sign-in popup:

In [ ]:
from google.colab import auth

auth.authenticate_user()
print("✅ Authenticated")

Now tell gcloud which project to use, then enable the APIs we need (Compute, GKE, networking):

In [ ]:
import os

# 👇 CHANGE THIS to your own project ID
PROJECT_ID = "your-project-id"  # @param {type:"string"}

os.environ["PROJECT_ID"] = PROJECT_ID
!gcloud config set project "$PROJECT_ID"
!gcloud config list project

In [ ]:
# Enable the required Google Cloud APIs (takes a minute or two)
!gcloud services enable \
    cloudresourcemanager.googleapis.com \
    servicenetworking.googleapis.com \
    compute.googleapis.com \
    container.googleapis.com

print("✅ APIs enabled")

# 3. Set up a GKE cluster

Time to create the cluster that kubectl-ai will manage.

⏳ **Heads up:** cluster creation takes **5–10 minutes**. Kick it off, grab a coffee, and we'll keep talking while it provisions.

> 💰 Cost note: 2 × `e2-standard-4` nodes cost roughly a few cents per hour. We'll delete everything in the cleanup section at the end.

In [ ]:
import os

# Feel free to change the zone to one closer to you
CLUSTER_NAME = "my-gke-cluster"  # @param {type:"string"}
ZONE = "us-central1-a"  # @param {type:"string"}

os.environ["CLUSTER_NAME"] = CLUSTER_NAME
os.environ["ZONE"] = ZONE

# ⏳ This takes 5-10 minutes — be patient!
!gcloud container clusters create "$CLUSTER_NAME" \
    --num-nodes 2 \
    --machine-type e2-standard-4 \
    --zone "$ZONE"

While the cluster provisions, install `kubectl` and the GKE auth plugin into this Colab runtime:

In [ ]:
# Install kubectl and the GKE auth plugin in this Colab VM
!sudo apt-get update -qq
!sudo apt-get install -y -qq kubectl google-cloud-cli-gke-gcloud-auth-plugin

import os
os.environ["USE_GKE_GCLOUD_AUTH_PLUGIN"] = "True"

!kubectl version --client
print("✅ kubectl ready")

**Verify the cluster**

Once the cell above finishes, fetch the cluster credentials and confirm both nodes are `Ready`:

In [ ]:
# Fetch cluster credentials so kubectl can talk to it
!gcloud container clusters get-credentials "$CLUSTER_NAME" --zone "$ZONE"

# Verify: nodes should be Ready
!kubectl get nodes

# And the default namespaces should exist
!kubectl get namespaces

# 4. Install kubectl-ai

Now for the star of the show. We'll use the quick-install script to get the `kubectl-ai` CLI into this runtime:

In [ ]:
# Download and install the kubectl-ai CLI
!curl -sSL https://raw.githubusercontent.com/GoogleCloudPlatform/kubectl-ai/main/install.sh | bash

# Make sure it's on PATH for the rest of the notebook
import os
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

!which kubectl-ai
print("✅ kubectl-ai installed")

**Set your Gemini API key**

kubectl-ai supports many LLM providers (Gemini, Vertex AI, OpenAI, Ollama, ...). We'll use **Gemini**.

1. Get a free API key from [Google AI Studio](https://aistudio.google.com/apikey)
2. **Recommended:** store it as a Colab secret — click the 🔑 **Secrets** icon in the left sidebar, add a secret named `GEMINI_API_KEY`, paste your key, and enable *Notebook access*
3. Run the cell below (it falls back to a hidden input prompt if no secret is found)

In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    print("✅ GEMINI_API_KEY loaded from Colab secrets")
except Exception:
    os.environ["GEMINI_API_KEY"] = getpass("Paste your Gemini API key: ")
    print("✅ GEMINI_API_KEY set")

In [ ]:
# Quick check: kubectl-ai should now respond
!kubectl-ai --quiet "what version of kubernetes is this cluster running?" --skip-permissions

# 5. Using kubectl-ai — talk to your cluster

You can now interact with the cluster using **natural language**. 🎉

kubectl-ai has two modes:

| Mode | How | Where it shines |
|---|---|---|
| **Interactive** | `kubectl-ai` opens a `>>>` prompt | Terminals (Cloud Shell, your laptop) |
| **Single-shot** | `kubectl-ai --quiet "your request"` | Scripts, CI, and notebooks like this one |

Since notebook cells aren't interactive terminals, we'll use **single-shot mode** here. Watch how each query gets translated into a real `kubectl` command:

> ⚠️ We pass `--skip-permissions` so kubectl-ai doesn't pause to ask for confirmation. Handy for a workshop — **not recommended in production**.

In [ ]:
# Our first natural-language query — pick the model explicitly
!kubectl-ai --quiet --model gemini-2.5-flash "list pods in default namespace" --skip-permissions

In [ ]:
# Ask a question about the cluster in plain English
!kubectl-ai --quiet "how many pods are there in the kube-system namespace?" --skip-permissions

**Create resources with plain English**

Let's create a real deployment — no YAML, no flags to remember:

In [ ]:
# Create a deployment — kubectl-ai figures out the kubectl command
!kubectl-ai --quiet "create a deployment named nginx with 2 replicas" --skip-permissions

# Verify it worked
!kubectl get deployments

In [ ]:
# 🏋️ Exercise: your turn! Write your own natural-language query below.
# Ideas: "patch the nginx deployment to use image nginx:1.27",
#        "expose the nginx deployment on port 80 as a service",
#        "delete the nginx deployment"
MY_QUERY = "scale the nginx deployment to 3 replicas"  # @param {type:"string"}

!kubectl-ai --quiet "$MY_QUERY" --skip-permissions

## 5.1 Troubleshooting with kubectl-ai 🔍

This is where kubectl-ai really shines. Let's **break something on purpose** and watch it diagnose the problem:

In [ ]:
# Deploy a pod with a deliberately broken image name
!kubectl run broken-app --image=nginx:doesnotexist
!sleep 15 && kubectl get pods

In [ ]:
# Now ask kubectl-ai to figure out what's wrong — like a junior SRE would
!kubectl-ai --quiet "the broken-app pod is not running. investigate why and explain the root cause" --skip-permissions

In [ ]:
# 🏋️ Exercise: ask kubectl-ai to FIX the broken pod
# Hint: try "fix the broken-app pod by setting its image to nginx:latest"
FIX_QUERY = ""  # @param {type:"string"}

!kubectl-ai --quiet "$FIX_QUERY" --skip-permissions
!kubectl get pods

# 6. (Optional) Interactive & chat modes

Colab cells are single-shot, but kubectl-ai has richer modes when you run it in a real terminal (Cloud Shell or your laptop):

**Interactive terminal mode**

```bash
kubectl-ai
```

This drops you into a `>>>` prompt where you can have a conversation with your cluster — context carries over between queries. Type `quit` to exit.

**Web chat UI**

```bash
kubectl-ai --llm-provider=gemini --ui-type=web --ui-listen-address=0.0.0.0:8080
```

In Cloud Shell, click **Web Preview → Preview on port 8080** to open a browser-based chat interface complete with query suggestions.

> 🏋️ **Take-home exercise:** open [Cloud Shell](https://shell.cloud.google.com), install kubectl-ai there (Section 4's install command works as-is), and try both modes against the same cluster.

# 7. Clean up 🧹

To avoid ongoing charges, delete the cluster when you're done:

In [ ]:
# ⚠️ This deletes the cluster and everything running on it
!gcloud container clusters delete "$CLUSTER_NAME" --zone "$ZONE" --quiet
print("✅ Cluster deleted — no more charges")

# 8. Congratulations! 🎉

You've used **kubectl-ai** to manage a real GKE cluster with natural language — querying, creating, scaling, breaking, diagnosing, and fixing Kubernetes resources.

**Keep exploring**

- 📖 [kubectl-ai on GitHub](https://github.com/GoogleCloudPlatform/kubectl-ai) — usage docs, supported providers, MCP mode
- ☸️ [GKE documentation](https://cloud.google.com/kubernetes-engine/docs)
- 🔑 [Google AI Studio](https://aistudio.google.com) — for your Gemini API keys
- 💬 Try kubectl-ai with local models via [Ollama](https://ollama.com) — no API key needed

Thanks for following along! ⭐ the repo if this was useful.